In [17]:
# Cell 1: Environment & Path Setup
# We set up our environment and define our exact file paths.

# Cell 1
import pandas as pd
import numpy as np
import warnings
import os

# Production-level config
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x) # Prevent scientific notation

# Define paths
RAW_DATA_PATH = '../data/raw/online_retail_II.xlsx'
PROCESSED_DATA_PATH = '../data/processed/cleaned_online_retail.parquet'

print("Libraries loaded and paths configured.")

Libraries loaded and paths configured.


In [18]:
# Cell 2: Targeted Data Ingestion & Standardization
#Here, we specifically target the "Year 2010-2011" sheet. We also immediately #standardize the column names. In production, you never want spaces in your #column names because it breaks dot-notation later.

print("Loading 'Year 2010-2011' sheet. This will take a moment...")

# Read only the specified sheet
df_raw = pd.read_excel(RAW_DATA_PATH, sheet_name="Year 2010-2011")

# Standardize column names across the board
df_raw.rename(columns={
    'Invoice': 'InvoiceNo',
    'Customer ID': 'CustomerID',
    'Price': 'UnitPrice'
}, inplace=True)

print(f"Raw Data Shape: {df_raw.shape}")
display(df_raw.head())

Loading 'Year 2010-2011' sheet. This will take a moment...
Raw Data Shape: (541910, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.550,17850.000,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.390,17850.000,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.750,17850.000,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.390,17850.000,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.390,17850.000,United Kingdom


In [19]:
# Cell 3: Missing Value Treatment
# For CLV and churn models, user attribution is everything. A transaction 
# without a CustomerID cannot be used to predict user behavior.

print("Missing values before treatment:")
print(df_raw.isnull().sum())

# Drop rows where CustomerID is missing
df_clean = df_raw.dropna(subset=['CustomerID']).copy()

print(f"\nDataset Shape after dropping null CustomerIDs: {df_clean.shape}")

Missing values before treatment:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Dataset Shape after dropping null CustomerIDs: (406830, 8)


In [20]:
"""
Cell 4: Anomaly Removal & Data Typing
Online retail datasets are notoriously messy. Cancellations are logged with a 'C' prefix in the invoice number, and sometimes there are data entry errors with negative prices or zero quantities. We isolate and remove these.

"""

# Cell 4
# Clean data types to ensure Parquet compatibility
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
df_clean['StockCode'] = df_clean['StockCode'].astype(str) 
# Convert ID to int first to remove the '.0' decimal, then to string
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str) 

# 1. Isolate and remove cancellations
cancellations = df_clean[df_clean['InvoiceNo'].str.startswith('C')]
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]

# 2. Filter out systemic errors (negative/zero quantity or price)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

print(f"Removed {len(cancellations)} cancelled transactions.")
print(f"Dataset Shape after anomaly removal: {df_clean.shape}")


Removed 8905 cancelled transactions.
Dataset Shape after anomaly removal: (397885, 8)


In [21]:
""" Cell 5: Base Feature Engineering & Assertion Checks
We calculate the financial value of each line item and format the dates. Finally, we add assert statements. If our cleaning logic failed, these assertions will throw an error immediately, preventing bad data from moving to Phase 2. 
"""
# Cell 5
# Calculate Total Amount per line item
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Ensure InvoiceDate is a datetime object
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Create a pure Date column (useful for daily cohorts later)
df_clean['Date'] = pd.to_datetime(df_clean['InvoiceDate'].dt.date)

# --- PRODUCTION PIPELINE CHECKS ---
assert df_clean['Quantity'].min() > 0, "Pipeline Error: Negative quantities detected!"
assert df_clean['TotalAmount'].min() > 0, "Pipeline Error: Negative amounts detected!"
assert df_clean['CustomerID'].isnull().sum() == 0, "Pipeline Error: Null IDs detected!"

print("All pipeline assertions passed.")
display(df_clean[['InvoiceNo', 'CustomerID', 'Date', 'TotalAmount']].head())

All pipeline assertions passed.


,InvoiceNo,CustomerID,Date,TotalAmount
0,536365,17850,2010-12-01,15.300
1,536365,17850,2010-12-01,20.340
2,536365,17850,2010-12-01,22.000
3,536365,17850,2010-12-01,20.340
4,536365,17850,2010-12-01,20.340


In [ ]:
"""
Cell 6: Persisting the Cleaned State
We save this strictly as a Parquet file.
"""
# Cell 6
# Ensure the processed directory exists just in case
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)

# Save to processed folder
df_clean.to_parquet(PROCESSED_DATA_PATH, index=False)
print(f"Cleaned, verified data successfully saved to: {PROCESSED_DATA_PATH}")

Cleaned, verified data successfully saved to: ../data/processed/cleaned_online_retail.parquet


: 